In [24]:
import jax 
import jax.numpy as jnp
import jax.random as jrandom
import jax.scipy.stats as stats

In [2]:
from probjax.core.custom_primitives.random_variable import rv_p

In [229]:
sampling_to_log_prob = {
    '_normal': lambda x: stats.norm.logpdf(x).sum(),
    '_uniform': lambda x,a,b: jax.scipy.stats.uniform.logpdf(x, a, (b-a)).sum(),
    '_bernoulli':  lambda x, p: jax.scipy.stats.bernoulli.logpmf(x,p).sum(),
    '_binomial': lambda x, n, p: jax.scipy.stats.binom.logpmf(x, n, p).sum(),
    '_gamma': lambda x, a: jax.scipy.stats.gamma.logpdf(x, a).sum(),
    '_exponential': lambda x: jax.scipy.stats.expon.logpdf(x).sum(),
    '_poisson': lambda x, a: jax.scipy.stats.poisson.logpmf(x, a).sum(),
    '_geometric': lambda x, p: jax.scipy.stats.geom.logpmf(x, p).sum(),
    '_cauchy': lambda x, a, b: jax.scipy.stats.cauchy.logpdf(x, a, b).sum(),
    '_pareto': lambda x, b: jax.scipy.stats.pareto.logpdf(x, b).sum(),
    '_chisquare': lambda x, df: jax.scipy.stats.chi2.logpdf(x, df).sum(),
    '_dirichlet': lambda x, a: jax.scipy.stats.dirichlet.logpdf(x, a).sum(),
    '_truncated_normal': lambda x, a, b: jax.scipy.stats.truncnorm.logpdf(x, a, b).sum(),
    '_multivariate_normal' : lambda x, mean, cov: jax.scipy.stats.multivariate_normal.logpdf(x, mean, cov).sum(),
    '_laplace': lambda x: jax.scipy.stats.laplace.logpdf(x).sum(),
    '_logistic': lambda x: jax.scipy.stats.logistic.logpdf(x).sum(),
    '_gumbel': lambda x:  NotImplementedError,
    '_maxwell': lambda x:  NotImplementedError,
    '_double_sided_maxwell': lambda x, loc,scale:  NotImplementedError,
    '_f': lambda x, df1, df2:  NotImplementedError,
    '_t': lambda x, df:  NotImplementedError,
    '_rademacher': lambda x:  NotImplementedError,
    '_wald': lambda x, a:  NotImplementedError,
}

In [176]:
?jax.random.

Signature:
jax.random.loggamma(
    key: 'KeyArrayLike',
    a: 'RealArray',
    shape: 'Shape | None' = None,
    dtype: 'DTypeLikeFloat' = <class 'float'>,
) -> 'Array'
Docstring:
Sample log-gamma random values with given shape and float dtype.

This function is implemented such that the following will hold for a
dtype-appropriate tolerance::

  np.testing.assert_allclose(jnp.exp(loggamma(*args)), gamma(*args), rtol=rtol)

The benefit of log-gamma is that for samples very close to zero (which occur frequently
when `a << 1`) sampling in log space provides better precision.

Args:
  key: a PRNG key used as the random key.
  a: a float or array of floats broadcast-compatible with ``shape``
    representing the parameter of the distribution.
  shape: optional, a tuple of nonnegative integers specifying the result
    shape. Must be broadcast-compatible with ``a``. The default (None)
    produces a result shape equal to ``a.shape``.
  dtype: optional, a float dtype for the returned values

In [194]:
jax.random.

AttributeError: module 'jax.scipy.stats' has no attribute 'gumbel_r'

In [237]:
def f(k):
    return jax.random.wald(k, 1)

In [238]:
f(jax.random.PRNGKey(0))

Array(0.866, dtype=float32)

In [239]:
jax.make_jaxpr(f)(jax.random.key(0))

{ lambda ; a:key<fry>[]. let
    b:f32[] = pjit[
      name=_wald
      jaxpr={ lambda ; c:key<fry>[] d:i32[]. let
          e:key<fry>[2] = random_split[shape=(2,)] c
          f:key<fry>[1] = slice[
            limit_indices=(1,)
            start_indices=(0,)
            strides=(1,)
          ] e
          g:key<fry>[] = squeeze[dimensions=(0,)] f
          h:key<fry>[1] = slice[
            limit_indices=(2,)
            start_indices=(1,)
            strides=(1,)
          ] e
          i:key<fry>[] = squeeze[dimensions=(0,)] h
          j:f32[] = convert_element_type[new_dtype=float32 weak_type=False] d
          k:f32[] = pjit[
            name=_normal
            jaxpr={ lambda ; l:key<fry>[]. let
                m:f32[] = pjit[
                  name=_normal_real
                  jaxpr={ lambda ; n:key<fry>[]. let
                      o:f32[] = pjit[
                        name=_uniform
                        jaxpr={ lambda ; p:key<fry>[] q:f32[] r:f32[]. let
            

In [62]:
jax.make_jaxpr(f)(jax.random.key(0)).consts

[array([1], dtype=int32), array([0], dtype=int32)]

In [4]:
def g(key):
    x1 = jax.random.normal(key)
    x2 = jax.random.uniform(key)
    x3 = jax.random.bernoulli(key)
    #x4 = jax.random.gamma(key, 2.)
    #y = x1 + x3 #+ x2 #+ x3 + x4
    return x3

In [5]:
jaxpr = jax.make_jaxpr(g)(jrandom.PRNGKey(0))

In [6]:
from jax.core import JaxprEqn

def update_eqn(eqn, name):
    sampling_fn_jaxpr = eqn.params["jaxpr"]
    sampling_name = eqn.params["name"]
    
    #print(sampling_fn_jaxpr.jaxpr.constvars)
    try:
        log_prob_fn = sampling_to_log_prob[sampling_name]
    except KeyError:
        raise NotImplementedError(f"Sampling function {sampling_name} no log_prob implemented")
    invals = jax._src.core.safe_map(lambda x: x.aval, eqn.outvars)
    additional_invals = jax._src.core.safe_map(lambda x: x.aval, sampling_fn_jaxpr.jaxpr.invars[1:])
    print(invals, additional_invals)
    log_prob_fn_jaxpr = jax.make_jaxpr(log_prob_fn)(*invals, *additional_invals)
    # print(log_prob_fn_jaxpr)
    
    params = {"sampling_fn_jaxpr": sampling_fn_jaxpr, "log_prob_fn_jaxpr": log_prob_fn_jaxpr, "name": name}
    new_eqn = JaxprEqn(eqn.invars, eqn.outvars, rv_p, params, eqn.effects, eqn.source_info)
    return new_eqn

def trace_rv(jaxpr):
    for i in range(len(jaxpr.eqns)):
        eqn = jaxpr.eqns[i]
        if eqn.primitive is not rv_p and "name" in eqn.params:
            new_eqn = update_eqn(eqn, "x" + str(i))
            jaxpr.eqns[i] = new_eqn
    return jaxpr

In [11]:
from probjax.core import joint_sample, log_potential_fn

In [12]:
jaxpr = jax.make_jaxpr(g)(jrandom.PRNGKey(0))

In [13]:
new_jaxpr = trace_rv(jaxpr)

[ShapedArray(float32[])] []
[ShapedArray(float32[])] [ShapedArray(float32[], weak_type=True), ShapedArray(float32[], weak_type=True)]
[ShapedArray(bool[])] [ShapedArray(float32[])]


In [17]:
f_traced = jax.core.jaxpr_as_fun(new_jaxpr)

In [18]:
f_traced(jrandom.PRNGKey(0))

[Array(True, dtype=bool)]

In [19]:
from probjax.core import joint_sample, log_potential_fn

In [20]:
sampler = joint_sample(f_traced)

samples = sampler(jrandom.PRNGKey(0))

In [21]:
samples

{'x1': Array(-0.206, dtype=float32),
 'x3': Array(0.418, dtype=float32),
 'x5': Array(True, dtype=bool)}

In [22]:
log_potential_fn(f_traced)(**samples)

Array(-1.633, dtype=float32)

In [50]:
jaxpr = jax.make_jaxpr(log_potential_fn(f_traced))(x1=1.)

KeyError: 'x3'